<a href="https://colab.research.google.com/github/Guruvarshin/SupportLM/blob/main/SupportLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Runtime check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Mon Jun 22 01:12:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Imports and setup

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install wandb anthropic datasets

In [ ]:
import torch
import unsloth
import transformers
import peft
import trl
import bitsandbytes
import wandb
import anthropic

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All imports successful")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
All imports successful


In [ ]:
from google.colab import userdata
import wandb
from huggingface_hub import login

wandb.login(key=userdata.get('WANDB_KEY'))
login(token=userdata.get('HF_TOKEN'))
anthropic_key = userdata.get('ANTHROPIC_KEY')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: guruprinting2003 (guruprinting2003-national-engineering-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# The shared folder may appear under a different path
import os
for root, dirs, files in os.walk('/content/drive'):
    if 'SupportLM' in dirs:
        print(os.path.join(root, 'SupportLM'))
        break

/content/drive/MyDrive/SupportLM


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/SupportLM/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/SupportLM/data', exist_ok=True)
print("Drive mounted. Folders created.")

## Data Generation

In [ ]:
import anthropic
import json
import random
import time
from collections import defaultdict
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_KEY'))

ISSUE_TYPES = [
    "order_status", "return_request", "payment_issue",
    "delivery_complaint", "product_defect", "cancellation_request",
    "wrong_item", "discount_query"
]

LANGUAGE_STYLES = {
    "hinglish": {
        "description": "Hindi-English code-switched, romanized Hindi (no Devanagari script)",
        "examples": [
            "mere order ka kya hua yaar 3 din ho gaye",
            "bhai payment cut ho gayi but order show nahi kar raha",
            "return karna hai please help karo"
        ]
    },
    "tanglish": {
        "description": "Tamil-English code-switched, romanized Tamil",
        "examples": [
            "enna da ithu order vandhe illa",
            "paisa pochi order illa bro",
            "return pannanuma sollunga"
        ]
    }
}

VARIANCE_CONFIGS = [
    {"anger": "frustrated but polite",        "length": "short (5-10 words)",   "typos": False, "emoji": False, "has_order_id": True},
    {"anger": "very angry",                   "length": "medium (15-25 words)", "typos": True,  "emoji": True,  "has_order_id": True},
    {"anger": "desperate and worried",        "length": "long (30-45 words)",   "typos": False, "emoji": True,  "has_order_id": False},
    {"anger": "casual and chill",             "length": "short (5-10 words)",   "typos": True,  "emoji": False, "has_order_id": False},
    {"anger": "very angry",                   "length": "short (5-10 words)",   "typos": True,  "emoji": True,  "has_order_id": True},
    {"anger": "frustrated but polite",        "length": "long (30-45 words)",   "typos": False, "emoji": False, "has_order_id": False},
    {"anger": "sarcastic",                    "length": "medium (15-25 words)", "typos": False, "emoji": True,  "has_order_id": True},
    {"anger": "first time complaining",       "length": "medium (15-25 words)", "typos": False, "emoji": False, "has_order_id": False},
    {"anger": "extremely furious",            "length": "long (30-45 words)",   "typos": True,  "emoji": True,  "has_order_id": True},
    {"anger": "tired and resigned",           "length": "short (5-10 words)",   "typos": False, "emoji": False, "has_order_id": False},
]

def generate_order_id():
    prefixes = ["ORD", "ORDER", "OD", "FLK", "AMZ", "MYN"]
    return f"{random.choice(prefixes)}{random.randint(100000, 999999)}"

def generate_batch(issue_type, language, variance):
    style = LANGUAGE_STYLES[language]
    order_id = generate_order_id() if variance["has_order_id"] else None

    order_id_instruction = (
        f'Include order ID "{order_id}" naturally in the message.'
        if order_id else
        "Do NOT include any order ID in the message."
    )

    prompt = f"""Generate 5 customer support messages for an Indian e-commerce app.

Issue type: {issue_type}
Language style: {language} — {style['description']}
Reference examples:
{chr(10).join(f'- "{e}"' for e in style['examples'])}

Message characteristics:
- Emotional tone: {variance['anger']}
- Length: {variance['length']}
- Typos: {'Yes — realistic ones like "ordar", "recieve", "cancle", "paymant"' if variance['typos'] else 'No'}
- Emojis: {'Yes — 1-2 like 😤 🙏 😢 😡 💔' if variance['emoji'] else 'No'}
- Order ID: {order_id_instruction}

For each message write an ideal response in the SAME language style. Response must:
- Be empathetic and acknowledge the customer's emotion
- Match the customer's code-switching pattern exactly
- Give a concrete next step
- Be 2-3 sentences maximum

Return ONLY a JSON array of 5 objects with fields:
"message", "issue_type", "order_id", "language", "response"

issue_type must be "{issue_type}", language must be "{language}", order_id must be {f'"{order_id}"' if order_id else 'null'}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )

    text = response.content[0].text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

# Generate all data — 2 batches per variance config = 1600 total examples
# 8 issues × 2 languages × 10 variance configs × 2 batches × 5 examples = 1600
all_data = []
total_calls = len(ISSUE_TYPES) * 2 * len(VARIANCE_CONFIGS) * 2
call_count = 0

for issue_type in ISSUE_TYPES:
    for language in ["hinglish", "tanglish"]:
        for variance in VARIANCE_CONFIGS:
            for batch_num in range(2):
                try:
                    batch = generate_batch(issue_type, language, variance)
                    all_data.extend(batch)
                    call_count += 1
                    print(f"[{call_count}/{total_calls}] {issue_type}/{language} — {len(all_data)} examples")
                    time.sleep(0.3)
                except Exception as e:
                    print(f"Error [{call_count}/{total_calls}] {issue_type}/{language}: {e}")
                    time.sleep(2)

print(f"\nGeneration complete. Total: {len(all_data)} examples")

# Split 80/10/10 stratified by issue_type + language
random.seed(42)
buckets = defaultdict(list)
for ex in all_data:
    buckets[f"{ex['issue_type']}_{ex['language']}"].append(ex)

train, val, test = [], [], []
for examples in buckets.values():
    random.shuffle(examples)
    n = len(examples)
    n_test = int(n * 0.10)
    n_val  = int(n * 0.10)
    test.extend(examples[:n_test])
    val.extend(examples[n_test:n_test + n_val])
    train.extend(examples[n_test + n_val:])

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# Save all three splits
for split, name in [(train, 'train'), (val, 'val'), (test, 'test')]:
    path = f'/content/drive/MyDrive/SupportLM/data/{name}.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(split, f, ensure_ascii=False, indent=2)
    print(f"Saved {name}.json — {len(split)} examples")

[1/320] order_status/hinglish — 5 examples
[2/320] order_status/hinglish — 10 examples
[3/320] order_status/hinglish — 15 examples
[4/320] order_status/hinglish — 20 examples
[5/320] order_status/hinglish — 25 examples
[6/320] order_status/hinglish — 30 examples
[7/320] order_status/hinglish — 35 examples
[8/320] order_status/hinglish — 40 examples
[9/320] order_status/hinglish — 45 examples
[10/320] order_status/hinglish — 50 examples
[11/320] order_status/hinglish — 55 examples
[12/320] order_status/hinglish — 60 examples
[13/320] order_status/hinglish — 65 examples
[14/320] order_status/hinglish — 70 examples
[15/320] order_status/hinglish — 75 examples
[16/320] order_status/hinglish — 80 examples
[17/320] order_status/hinglish — 85 examples
[18/320] order_status/hinglish — 90 examples
[19/320] order_status/hinglish — 95 examples
[20/320] order_status/hinglish — 100 examples
[21/320] order_status/tanglish — 105 examples
[22/320] order_status/tanglish — 110 examples
[23/320] order_st

In [ ]:
import json
from collections import Counter

with open('/content/drive/MyDrive/SupportLM/data/train.json', 'r') as f:
    train = json.load(f)

print(f"Train count: {len(train)}")
issue_counts = Counter(d['issue_type'] for d in train)
lang_counts = Counter(d['language'] for d in train)
print("Issues:", dict(issue_counts))
print("Languages:", dict(lang_counts))

Train count: 1277
Issues: {'order_status': 160, 'return_request': 160, 'payment_issue': 160, 'delivery_complaint': 160, 'product_defect': 157, 'cancellation_request': 160, 'wrong_item': 160, 'discount_query': 160}
Languages: {'hinglish': 637, 'tanglish': 640}


## Baseline evaluation

In [ ]:
%%capture
!pip install openai

In [ ]:
import json
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_KEY'))

with open('/content/drive/MyDrive/SupportLM/data/test.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test examples")

Loaded 159 test examples


In [ ]:
SYSTEM_PROMPT = """You are a customer support agent for an Indian e-commerce platform.
A customer will send you a message in Hinglish (Hindi-English) or Tanglish (Tamil-English).

You must respond with ONLY a JSON object with exactly these three fields:
{
  "issue_type": one of [order_status, return_request, payment_issue, delivery_complaint, product_defect, cancellation_request, wrong_item, discount_query],
  "order_id": the order ID if mentioned in the message (e.g. ORD123, AMZ456) or null,
  "response": an empathetic reply in the same language the customer used
}

Return only the JSON object, nothing else."""

print("System prompt ready")

System prompt ready


In [ ]:
results = []
errors = 0

for i, example in enumerate(test_data):
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": example["message"]}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )

        pred = json.loads(completion.choices[0].message.content)

        results.append({
            "message":         example["message"],
            "language":        example["language"],
            "true_issue_type": example["issue_type"],
            "true_order_id":   example["order_id"],
            "true_response":   example["response"],
            "pred_issue_type": pred.get("issue_type", ""),
            "pred_order_id":   pred.get("order_id", None),
            "pred_response":   pred.get("response", ""),
        })

        if (i + 1) % 20 == 0:
            print(f"[{i+1}/159] done")

    except Exception as e:
        print(f"Error on example {i}: {e}")
        errors += 1

print(f"\nCompleted {len(results)}/159 examples. Errors: {errors}")

[20/159] done
[40/159] done
[60/159] done
[80/159] done
[100/159] done
[120/159] done
[140/159] done

Completed 159/159 examples. Errors: 0


In [ ]:
correct_class = sum(
    1 for r in results
    if r["pred_issue_type"].strip().lower() == r["true_issue_type"].strip().lower()
)
class_accuracy = correct_class / len(results) * 100

print(f"Classification accuracy: {class_accuracy:.1f}%  ({correct_class}/{len(results)})")

Classification accuracy: 74.8%  (119/159)


In [ ]:
order_id_present = [r for r in results if r["true_order_id"] is not None]
order_id_absent  = [r for r in results if r["true_order_id"] is None]

correct_present = sum(
    1 for r in order_id_present
    if r["pred_order_id"] and
    r["true_order_id"].upper() in str(r["pred_order_id"]).upper()
)
correct_absent = sum(
    1 for r in order_id_absent
    if not r["pred_order_id"]
)

order_id_accuracy = (correct_present + correct_absent) / len(results) * 100

print(f"Order ID extraction accuracy: {order_id_accuracy:.1f}%")
print(f"  Correctly found when present: {correct_present}/{len(order_id_present)}")
print(f"  Correctly null when absent:   {correct_absent}/{len(order_id_absent)}")

Order ID extraction accuracy: 98.7%
  Correctly found when present: 70/72
  Correctly null when absent:   87/87


In [ ]:
def language_matches(message, response):
    hinglish_words = ["yaar", "bhai", "kya", "nahi", "kar", "ho", "gaya", "mera", "bilkul", "samajh"]
    tanglish_words = ["da", "bro", "pannu", "sollu", "illa", "vandhuthu", "enna", "pochi", "aaguthu"]
    msg = message.lower()
    res = response.lower()
    if any(w in msg for w in hinglish_words):
        return any(w in res for w in hinglish_words)
    if any(w in msg for w in tanglish_words):
        return any(w in res for w in tanglish_words)
    return True

lang_match = sum(1 for r in results if language_matches(r["message"], r["pred_response"]))
lang_match_rate = lang_match / len(results) * 100

print(f"Language match rate: {lang_match_rate:.1f}%  ({lang_match}/{len(results)})")

Language match rate: 96.2%  (153/159)


In [ ]:
print("\n" + "="*50)
print("GPT-4o-mini BASELINE RESULTS")
print("="*50)
print(f"Classification accuracy : {class_accuracy:.1f}%")
print(f"Order ID extraction acc : {order_id_accuracy:.1f}%")
print(f"Language match rate     : {lang_match_rate:.1f}%")
print("="*50)

with open('/content/drive/MyDrive/SupportLM/data/baseline_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        "model": "gpt-4o-mini",
        "classification_accuracy": class_accuracy,
        "order_id_accuracy": order_id_accuracy,
        "language_match_rate": lang_match_rate,
        "per_example": results
    }, f, ensure_ascii=False, indent=2)

print("\nSaved to baseline_results.json")


GPT-4o-mini BASELINE RESULTS
Classification accuracy : 74.8%
Order ID extraction acc : 98.7%
Language match rate     : 96.2%

Saved to baseline_results.json


## Full Fine-tune Attempt

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading model in float16 (no quantization)...")
print(f"VRAM before load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True
)

print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM remaining: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM after load: 6.18 GB
VRAM remaining: 9.46 GB
Model parameters: 3.09B


In [ ]:
import torch
from torch.optim import AdamW

model.train()
optimizer = AdamW(model.parameters(), lr=2e-5)

print("Attempting real training step with Adam optimizer...")
print("Adam needs 2 extra copies of all gradients = ~12GB extra")
print(f"VRAM before optimizer step: {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

try:
    for i in range(5):
        input_ids = format_example(train_data[i])
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        print(f"Example {i+1} backward done — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    print("\nCalling optimizer.step() — this is where Adam materializes...")
    optimizer.step()
    print("If you see this, memory was somehow enough (very unexpected)")

except RuntimeError as e:
    print(f"\nCRASHED at: {e}")
    print("\nThis is exactly the lesson.")
    print("Adam optimizer states = 2x model size = no room on T4.")

finally:
    optimizer.zero_grad(set_to_none=True)

Attempting real training step with Adam optimizer...
Adam needs 2 extra copies of all gradients = ~12GB extra
VRAM before optimizer step: 12.37 GB

Example 1 backward done — VRAM: 12.37 GB
Example 2 backward done — VRAM: 12.37 GB
Example 3 backward done — VRAM: 12.37 GB
Example 4 backward done — VRAM: 12.37 GB
Example 5 backward done — VRAM: 12.37 GB

Calling optimizer.step() — this is where Adam materializes...

CRASHED at: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.36 GiB is allocated by PyTorch, and 60.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-all

In [ ]:
import gc
import torch

try:
    del model
    del optimizer
except: pass
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.\n")
print("="*55)
print("WHY FULL FINE-TUNING FAILS ON T4")
print("="*55)
print(f"Model weights  (3B × float16)      =  ~6 GB")
print(f"Gradients      (3B × float16)      =  ~6 GB")
print(f"Adam momentum  (3B × float32)      =  ~12 GB")
print(f"Adam variance  (3B × float32)      =  ~12 GB")
print(f"                          TOTAL    = ~36 GB")
print(f"T4 VRAM available                  = 15.6 GB")
print("="*55)
print("\nLoRA solution:")
print("Only train 2 small matrices per layer (rank=16)")
print("That is ~24M trainable params instead of 3B")
print("= 0.8% of the model — fits easily on T4")

GPU memory cleared.

WHY FULL FINE-TUNING FAILS ON T4
Model weights  (3B × float16)      =  ~6 GB
Gradients      (3B × float16)      =  ~6 GB
Adam momentum  (3B × float32)      =  ~12 GB
Adam variance  (3B × float32)      =  ~12 GB
                          TOTAL    = ~36 GB
T4 VRAM available                  = 15.6 GB

LoRA solution:
Only train 2 small matrices per layer (rank=16)
That is ~24M trainable params instead of 3B
= 0.8% of the model — fits easily on T4


## QLoRA + Unsloth SFT

In [ ]:
import torch
from unsloth import FastLanguageModel
# try:
#     del model
#     del tokenizer
# except: pass
# gc.collect()
# torch.cuda.empty_cache()

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")
print("Model loaded successfully")

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
VRAM used: 2.40 GB
VRAM free: 13.24 GB
Model loaded successfully


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M")
print(f"Total parameters:     {total/1e6:.1f}M")
print(f"Trainable %:          {100*trainable/total:.2f}%")

NameError: name 'model' is not defined

In [ ]:
import json

def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response."
        },
        {
            "role": "user",
            "content": example["message"]
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "issue_type": example["issue_type"],
                "order_id":   example["order_id"],
                "response":   example["response"]
            }, ensure_ascii=False)
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

# Load train and val data
with open('/content/drive/MyDrive/SupportLM/data/train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open('/content/drive/MyDrive/SupportLM/data/val.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

# Preview one formatted example so you can see the template
print("FORMATTED EXAMPLE:")
print("-"*60)
print(format_example(train_data[0]))
print("-"*60)
print(f"\nTrain examples: {len(train_data)}")
print(f"Val examples:   {len(val_data)}")

FORMATTED EXAMPLE:
------------------------------------------------------------
<|im_start|>system
You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response.<|im_end|>
<|im_start|>user
mere ordar ka kya hua yaar 3 din<|im_end|>
<|im_start|>assistant
{"issue_type": "order_status", "order_id": null, "response": "bilkul bhai samajh rahe hain frustration, ab tak paack nahi hua kya? Ek dum check kar dete hain aur 2 hours mein update de denge."}<|im_end|>

------------------------------------------------------------

Train examples: 1277
Val examples:   159


In [ ]:
from datasets import Dataset

def to_dataset(data):
    return Dataset.from_dict({
        "text": [format_example(ex) for ex in data]
    })

train_dataset = to_dataset(train_data)
val_dataset   = to_dataset(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")
print("\nSample text field (first 200 chars):")
print(train_dataset[0]["text"][:200])

Train dataset: Dataset({
    features: ['text'],
    num_rows: 1277
})
Val dataset:   Dataset({
    features: ['text'],
    num_rows: 159
})

Sample text field (first 200 chars):
<|im_start|>system
You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response.<|im_end|>
<|i


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch
import shutil
import os

# incomplete = "/content/drive/MyDrive/SupportLM/checkpoints/checkpoint-100"
# if os.path.exists(incomplete):
#     shutil.rmtree(incomplete)
#     print("Deleted incomplete checkpoint-100")

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/SupportLM/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="wandb",
    run_name="supportlm-sft-qwen2.5-3b",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
)

print("Starting training from scratch...")
print("W&B link will appear below\n")

trainer.train()

print("\nTraining complete")

Deleted incomplete checkpoint-100


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1277 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/159 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training from scratch...
W&B link will appear below



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,277 | Num Epochs = 3 | Total steps = 240
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,Validation Loss
100,0.930516,1.087191
200,0.682665,1.098944
240,0.729496,1.090426


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportLM/checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportLM/checkpoints/checkpoint-240/tokenizer_config.json.



Training complete


In [ ]:
ADAPTER_PATH = "/content/drive/MyDrive/SupportLM/sft-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"Adapter saved to {ADAPTER_PATH}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportLM/sft-adapter/tokenizer_config.json.


Adapter saved to /content/drive/MyDrive/SupportLM/sft-adapter


## Evaluate SFT vs GPT-4o-mini baseline

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
print("Model switched to inference mode")

Model switched to inference mode


In [ ]:
import json
import torch
from google.colab import userdata

with open('/content/drive/MyDrive/SupportLM/data/test.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

SYSTEM_PROMPT = "You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response."

def run_inference(message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": message}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens, not the input
    new_tokens = output_ids[0][input_ids.shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response_text.strip()

def parse_output(text):
    try:
        # Handle cases where model wraps output in markdown code blocks
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except:
        return {"issue_type": "", "order_id": None, "response": ""}

results = []
errors = 0

for i, example in enumerate(test_data):
    try:
        raw = run_inference(example["message"])
        pred = parse_output(raw)

        results.append({
            "message":         example["message"],
            "language":        example["language"],
            "true_issue_type": example["issue_type"],
            "true_order_id":   example["order_id"],
            "true_response":   example["response"],
            "pred_issue_type": pred.get("issue_type", ""),
            "pred_order_id":   pred.get("order_id", None),
            "pred_response":   pred.get("response", ""),
            "raw_output":      raw,
        })

        if (i + 1) % 20 == 0:
            print(f"[{i+1}/159] done")

    except Exception as e:
        print(f"Error on example {i}: {e}")
        errors += 1

print(f"\nCompleted {len(results)}/159 examples. Errors: {errors}")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

[20/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

[40/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

[120/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

[140/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Completed 159/159 examples. Errors: 0


In [ ]:
# Classification accuracy
correct_class = sum(
    1 for r in results
    if r["pred_issue_type"].strip().lower() == r["true_issue_type"].strip().lower()
)
class_accuracy = correct_class / len(results) * 100

In [ ]:
# Order ID extraction accuracy
order_id_present = [r for r in results if r["true_order_id"] is not None]
order_id_absent  = [r for r in results if r["true_order_id"] is None]

correct_present = sum(
    1 for r in order_id_present
    if r["pred_order_id"] and
    r["true_order_id"].upper() in str(r["pred_order_id"]).upper()
)
correct_absent = sum(
    1 for r in order_id_absent
    if not r["pred_order_id"]
)
order_id_accuracy = (correct_present + correct_absent) / len(results) * 100

In [ ]:
# Language match rate
def language_matches(message, response):
    hinglish_words = ["yaar", "bhai", "kya", "nahi", "kar", "ho", "gaya", "mera", "bilkul", "samajh"]
    tanglish_words = ["da", "bro", "pannu", "sollu", "illa", "vandhuthu", "enna", "pochi", "aaguthu"]
    msg = message.lower()
    res = response.lower()
    if any(w in msg for w in hinglish_words):
        return any(w in res for w in hinglish_words)
    if any(w in msg for w in tanglish_words):
        return any(w in res for w in tanglish_words)
    return True

lang_match = sum(1 for r in results if language_matches(r["message"], r["pred_response"]))
lang_match_rate = lang_match / len(results) * 100

In [ ]:
# Print comparison table
print("\n" + "="*55)
print(f"{'Metric':<30} {'GPT-4o-mini':>10} {'SFT Model':>10}")
print("="*55)
print(f"{'Classification accuracy':<30} {'74.8%':>10} {class_accuracy:>9.1f}%")
print(f"{'Order ID extraction':<30} {'98.7%':>10} {order_id_accuracy:>9.1f}%")
print(f"{'Language match rate':<30} {'96.2%':>10} {lang_match_rate:>9.1f}%")
print("="*55)



Metric                         GPT-4o-mini  SFT Model
Classification accuracy             74.8%      79.9%
Order ID extraction                 98.7%      99.4%
Language match rate                 96.2%      99.4%


In [ ]:
# Save results
with open('/content/drive/MyDrive/SupportLM/data/sft_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        "model": "qwen2.5-3b-sft",
        "classification_accuracy": class_accuracy,
        "order_id_accuracy": order_id_accuracy,
        "language_match_rate": lang_match_rate,
        "per_example": results
    }, f, ensure_ascii=False, indent=2)

print("Saved to sft_results.json")

Saved to sft_results.json


## DPO (Direct Preference Optimization)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_training(model)

print("Model switched back to training mode")

NameError: name 'model' is not defined

In [ ]:
import json
import random
from datasets import Dataset

with open('/content/drive/MyDrive/SupportLM/data/train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Templates for bad English responses
# These are deliberately generic and unhelpful
ENGLISH_REJECTIONS = {
    "order_status":         "Your order is currently being processed. Please wait for updates.",
    "return_request":       "Please visit our website to initiate a return request.",
    "payment_issue":        "Your payment is being verified. Please contact your bank.",
    "delivery_complaint":   "We apologize for the delay. Our team will look into this.",
    "product_defect":       "Please send photos of the defective product to our email.",
    "cancellation_request": "Cancellation requests take 5-7 business days to process.",
    "wrong_item":           "We apologize for the inconvenience. Please return the item.",
    "discount_query":       "Discounts are applied automatically at checkout when applicable.",
}

def create_rejected_response(example):
    rejection_type = random.choice(["english", "vague"])

    if rejection_type == "english":
        return ENGLISH_REJECTIONS[example["issue_type"]]

    else:
        vague_responses = [
            "hum aapki problem dekh rahe hain, thoda wait karo.",
            "sorry for the inconvenience, we will look into it.",
            "please contact customer support for further assistance.",
            "naan check pannuven, wait pannunga.",
            "we understand your concern and will resolve it soon.",
        ]
        return random.choice(vague_responses)

SYSTEM_PROMPT = "You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response."

def format_prompt(message):
    return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{message}<|im_end|>\n<|im_start|>assistant\n"

def format_chosen(example):
    return json.dumps({
        "issue_type": example["issue_type"],
        "order_id":   example["order_id"],
        "response":   example["response"]
    }, ensure_ascii=False)

random.seed(42)

dpo_data = []
for example in train_data:
    dpo_data.append({
        "prompt":   format_prompt(example["message"]),
        "chosen":   format_chosen(example),
        "rejected": create_rejected_response(example),
    })

dpo_dataset = Dataset.from_list(dpo_data)

print(f"DPO dataset: {len(dpo_dataset)} preference pairs")
print("\nExample pair:")
print(f"Prompt:   {dpo_data[0]['prompt'][:80]}...")
print(f"Chosen:   {dpo_data[0]['chosen'][:80]}...")
print(f"Rejected: {dpo_data[0]['rejected'][:80]}...")

DPO dataset: 1277 preference pairs

Example pair:
Prompt:   <|im_start|>system
You are a customer support agent for an Indian e-commerce pla...
Chosen:   {"issue_type": "order_status", "order_id": null, "response": "bilkul bhai samajh...
Rejected: Your order is currently being processed. Please wait for updates....


In [ ]:
from trl import DPOTrainer, DPOConfig
import torch

dpo_config = DPOConfig(
    output_dir="/content/drive/MyDrive/SupportLM/dpo-checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=5e-5,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    logging_steps=10,

    beta=0.1,

    max_length=512,

    max_prompt_length=256,

    optim="adamw_8bit",

    lr_scheduler_type="cosine",

    report_to="wandb",
    run_name="supportlm-dpo-qwen2.5-3b",

    seed=42,
)

dpo_trainer = DPOTrainer(
    model=model,

    ref_model=None,

    train_dataset=dpo_dataset,

    tokenizer=tokenizer,

    args=dpo_config,
)

print("Starting DPO training...")

dpo_trainer.train()

print("\nDPO training complete")

Extracting prompt in train dataset (num_proc=2):   0%|          | 0/1277 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/1277 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1277 [00:00<?, ? examples/s]

Starting DPO training...
This will take approximately 10-15 minutes



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,277 | Num Epochs = 1 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,0.000618,17.647135,0.551108,1.000000,17.096027,-70.617111,-87.570335,-1.390468,-0.826861
20,0.000000,18.364996,-1.826613,1.000000,20.191608,-70.318710,-106.477188,-1.317482,-0.666064
30,0.000000,17.911703,-2.638170,1.000000,20.549871,-70.294266,-113.516075,-1.279087,-0.621257
40,0.000000,18.121508,-2.615037,1.000000,20.736546,-66.296913,-113.744606,-1.301786,-0.617390
50,0.000000,18.853123,-2.670765,1.000000,21.523884,-67.976364,-116.478760,-1.287214,-0.659660
60,0.000000,18.731667,-2.784065,1.000000,21.515732,-67.930725,-118.016930,-1.294155,-0.691418
70,0.000000,18.815041,-2.868176,1.000000,21.683216,-68.677353,-115.673500,-1.248670,-0.648817
80,0.000000,18.860409,-2.605093,1.000000,21.465502,-63.872589,-115.803528,-1.306373,-0.611601


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportLM/dpo-checkpoints/checkpoint-80/tokenizer_config.json.



DPO training complete


In [ ]:
DPO_ADAPTER_PATH = "/content/drive/MyDrive/SupportLM/dpo-adapter"

model.save_pretrained(DPO_ADAPTER_PATH)
tokenizer.save_pretrained(DPO_ADAPTER_PATH)

print(f"DPO adapter saved to {DPO_ADAPTER_PATH}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SupportLM/dpo-adapter/tokenizer_config.json.


DPO adapter saved to /content/drive/MyDrive/SupportLM/dpo-adapter


## DPO Evaluation & Final Results

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
print("Model switched to inference mode")

Model switched to inference mode


In [ ]:
# Login to HuggingFace and W&B, mount Drive, load DPO model
import torch
import json
from unsloth import FastLanguageModel
from huggingface_hub import login
from google.colab import userdata, drive

# Mount Google Drive — all your saved files are here
drive.mount('/content/drive')

# Login to HuggingFace so we can push the model later
login(token=userdata.get('HF_TOKEN'))

print("Loading DPO model from saved adapter...")

# Load the base model with 4-bit quantization
# Same settings as training
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

# Load the DPO adapter weights on top of the base model
# This is the adapter saved at the end of Phase 6
from peft import PeftModel
model = PeftModel.from_pretrained(
    model,
    "/content/drive/MyDrive/SupportLM/dpo-adapter"
)

print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("DPO model loaded successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading DPO model from saved adapter...
==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
VRAM used: 4.91 GB
DPO model loaded successfully


In [ ]:
from unsloth import FastLanguageModel

# Switch off gradient tracking for faster generation
FastLanguageModel.for_inference(model)
print("Model ready for inference")

Model ready for inference


In [ ]:
import json
import torch

with open('/content/drive/MyDrive/SupportLM/data/test.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

SYSTEM_PROMPT = "You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response."

def run_inference(message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": message}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def parse_output(text):
    try:
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except:
        return {"issue_type": "", "order_id": None, "response": ""}

results = []
errors = 0

for i, example in enumerate(test_data):
    try:
        raw  = run_inference(example["message"])
        pred = parse_output(raw)

        results.append({
            "message":         example["message"],
            "language":        example["language"],
            "true_issue_type": example["issue_type"],
            "true_order_id":   example["order_id"],
            "true_response":   example["response"],
            "pred_issue_type": pred.get("issue_type", ""),
            "pred_order_id":   pred.get("order_id", None),
            "pred_response":   pred.get("response", ""),
            "raw_output":      raw,
        })

        if (i + 1) % 20 == 0:
            print(f"[{i+1}/159] done")

    except Exception as e:
        print(f"Error on example {i}: {e}")
        errors += 1

print(f"\nCompleted {len(results)}/159 examples. Errors: {errors}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

[20/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

[40/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

[120/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

[140/159] done


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Completed 159/159 examples. Errors: 0


In [ ]:
with open('/content/drive/MyDrive/SupportLM/data/baseline_results.json', 'r') as f:
    baseline = json.load(f)

with open('/content/drive/MyDrive/SupportLM/data/sft_results.json', 'r') as f:
    sft = json.load(f)

In [ ]:
correct_class = sum(
    1 for r in results
    if r["pred_issue_type"].strip().lower() == r["true_issue_type"].strip().lower()
)
class_accuracy = correct_class / len(results) * 100

In [ ]:
order_id_present = [r for r in results if r["true_order_id"] is not None]
order_id_absent  = [r for r in results if r["true_order_id"] is None]

correct_present = sum(
    1 for r in order_id_present
    if r["pred_order_id"] and
    r["true_order_id"].upper() in str(r["pred_order_id"]).upper()
)
correct_absent = sum(
    1 for r in order_id_absent
    if not r["pred_order_id"]
)
order_id_accuracy = (correct_present + correct_absent) / len(results) * 100

In [ ]:
def language_matches(message, response):
    hinglish_words = ["yaar", "bhai", "kya", "nahi", "kar", "ho", "gaya", "mera", "bilkul", "samajh"]
    tanglish_words = ["da", "bro", "pannu", "sollu", "illa", "vandhuthu", "enna", "pochi", "aaguthu"]
    msg = message.lower()
    res = response.lower()
    if any(w in msg for w in hinglish_words):
        return any(w in res for w in hinglish_words)
    if any(w in msg for w in tanglish_words):
        return any(w in res for w in tanglish_words)
    return True

lang_match     = sum(1 for r in results if language_matches(r["message"], r["pred_response"]))
lang_match_rate = lang_match / len(results) * 100

In [ ]:
print("\n" + "="*65)
print(f"{'Metric':<30} {'GPT-4o-mini':>10} {'SFT':>10} {'DPO':>10}")
print("="*65)
print(f"{'Classification accuracy':<30} {baseline['classification_accuracy']:>9.1f}% {sft['classification_accuracy']:>9.1f}% {class_accuracy:>9.1f}%")
print(f"{'Order ID extraction':<30} {baseline['order_id_accuracy']:>9.1f}% {sft['order_id_accuracy']:>9.1f}% {order_id_accuracy:>9.1f}%")
print(f"{'Language match rate':<30} {baseline['language_match_rate']:>9.1f}% {sft['language_match_rate']:>9.1f}% {lang_match_rate:>9.1f}%")
print("="*65)


Metric                         GPT-4o-mini        SFT        DPO
Classification accuracy             74.8%      79.9%      78.0%
Order ID extraction                 98.7%      99.4%      99.4%
Language match rate                 96.2%      99.4%     100.0%


In [ ]:
with open('/content/drive/MyDrive/SupportLM/data/dpo_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        "model": "qwen2.5-3b-dpo",
        "classification_accuracy": class_accuracy,
        "order_id_accuracy": order_id_accuracy,
        "language_match_rate": lang_match_rate,
        "per_example": results
    }, f, ensure_ascii=False, indent=2)

print("Saved to dpo_results.json")

Saved to dpo_results.json


## Reload SFT model for merging

In [ ]:
import torch
import gc
from unsloth import FastLanguageModel
from peft import PeftModel

# Clear DPO model from memory first
try:
    del model
except: pass
gc.collect()
torch.cuda.empty_cache()

print("Loading base model fresh...")

# Load base model with 4-bit quantization
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

# Load the SFT adapter on top of base model
# SFT had better classification accuracy so we deploy this one
model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/SupportLM/sft-adapter"
)

print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("SFT model loaded successfully")

Loading base model fresh...
==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
VRAM used: 2.52 GB
SFT model loaded successfully


## Merge adapter into base model

In [ ]:
%%capture
!pip install transformers --upgrade --quiet

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import torch
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, HfApi
from google.colab import userdata, drive

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

HF_USERNAME = "GuruVarshini"
MODEL_NAME  = f"{HF_USERNAME}/supportlm-qwen2.5-3b"

print("Loading base model in float16 (no quantization this time)...")
print("This is needed for clean merging\n")

# Load base model in float16 WITHOUT 4-bit quantization
# 4-bit quantization prevents clean merging
# float16 uses more VRAM but allows proper weight merging
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu",
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)

print("Base model loaded on CPU")

# Load SFT adapter on top of the float16 base model
model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/SupportLM/sft-adapter",
    torch_dtype=torch.float16,
)

print("Adapter loaded")

# Merge adapter weights permanently into base model
# This runs on CPU so it is slower but avoids all GPU memory issues
print("Merging adapter into base model (on CPU)...")
model = model.merge_and_unload()

print("Merge complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading base model in float16 (no quantization this time)...
This is needed for clean merging



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Base model loaded on CPU
Adapter loaded
Merging adapter into base model (on CPU)...
Merge complete

Pushing to GuruVarshini/supportlm-qwen2.5-3b...
This will take 10-15 minutes



TypeError: PushToHubMixin.push_to_hub() got an unexpected keyword argument 'safe_serialization'

## Push merged model to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_USERNAME = "GuruVarshini"
MODEL_NAME  = f"{HF_USERNAME}/supportlm-qwen2.5-3b"

api = HfApi(token=userdata.get('HF_TOKEN'))
api.create_repo(
    repo_id=MODEL_NAME,
    repo_type="model",
    private=False,
    exist_ok=True,
)

print(f"Pushing to {MODEL_NAME}...")
print("This will take 10-15 minutes\n")

# Push model — removed safe_serialization which is not supported
model.push_to_hub(
    MODEL_NAME,
    token=userdata.get('HF_TOKEN'),
    max_shard_size="2GB",
)

tokenizer.push_to_hub(
    MODEL_NAME,
    token=userdata.get('HF_TOKEN'),
)

print(f"\nDone: https://huggingface.co/{MODEL_NAME}")

Pushing to GuruVarshini/supportlm-qwen2.5-3b...
This will take 10-15 minutes



Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00004.safetensors:   1%|          | 15.9MB / 1.99GB            

...

  ...0004-of-00004.safetensors:   0%|          |  524kB /  263MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpknonyj6u/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            


Done: https://huggingface.co/GuruVarshini/supportlm-qwen2.5-3b


In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get('HF_TOKEN'))

# Update the model card to include the pipeline_tag
# This tells HuggingFace Inference API what type of model this is
MODEL_CARD = """---
language:
- hi
- ta
- en
tags:
- customer-support
- hinglish
- tanglish
- fine-tuned
- qwen2.5
- qlora
base_model: Qwen/Qwen2.5-3B-Instruct
pipeline_tag: text-generation
---

# SupportLM — Hinglish/Tanglish Customer Support Model

A fine-tuned Qwen2.5-3B model for Indian e-commerce customer support
in code-switched language (Hinglish and Tanglish).

## What it does

Given a customer support message in Hinglish or Tanglish, the model:
1. Classifies the issue type into one of 8 categories
2. Extracts the order ID if present in the message
3. Generates an empathetic response in the same language the customer used

## Benchmark Results

Evaluated on 159 held-out test examples never seen during training.

| Metric | GPT-4o-mini | SFT Model | DPO Model |
|--------|-------------|-----------|-----------|
| Classification accuracy | 74.8% | **79.9%** | 78.0% |
| Order ID extraction | 98.7% | **99.4%** | 99.4% |
| Language match rate | 96.2% | 99.4% | **100.0%** |

SFT model chosen for deployment — best classification accuracy.

## Issue Types

order_status, return_request, payment_issue, delivery_complaint,
product_defect, cancellation_request, wrong_item, discount_query

## Training Details

- Base model: Qwen/Qwen2.5-3B-Instruct
- Training data: 1277 synthetic Hinglish/Tanglish examples
- Training method: QLoRA (r=16, alpha=32) + DPO (beta=0.1)
- Hardware: Google Colab T4 GPU (free tier)
- Training time: ~40 minutes total (SFT + DPO)

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import json

model = AutoModelForCausalLM.from_pretrained("GuruVarshini/supportlm-qwen2.5-3b")
tokenizer = AutoTokenizer.from_pretrained("GuruVarshini/supportlm-qwen2.5-3b")

message = "bhai mere order ka kya hua yaar 3 din ho gaye"

messages = [
    {"role": "system", "content": "You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response."},
    {"role": "user", "content": message}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)
output = model.generate(input_ids, max_new_tokens=200)
print(tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True))
"""

api.upload_file(
path_or_fileobj=MODEL_CARD.encode(),
path_in_repo="README.md",
repo_id="GuruVarshini/supportlm-qwen2.5-3b",
repo_type="model",
)

print("Model card updated with pipeline_tag: text-generation")

Model card updated with pipeline_tag: text-generation


In [ ]:
APP_CODE = '''
import gradio as gr
import json
import time
from openai import OpenAI
import os
import requests

MODEL_ID      = "GuruVarshini/supportlm-qwen2.5-3b"
SYSTEM_PROMPT = (
    "You are a customer support agent for an Indian e-commerce platform. "
    "Given a customer message, respond with a JSON object containing "
    "issue_type, order_id, and response."
)

def call_supportlm(message):
    token = os.environ.get("HF_TOKEN", "")

    full_prompt = (
        f"<|im_start|>system\\n{SYSTEM_PROMPT}<|im_end|>\\n"
        f"<|im_start|>user\\n{message}<|im_end|>\\n"
        f"<|im_start|>assistant\\n"
    )

    url = f"https://api-inference.huggingface.co/models/{MODEL_ID}"
    headers = {"Authorization": f"Bearer {token}"}
    payload = {
        "inputs": full_prompt,
        "parameters": {
            "max_new_tokens": 200,
            "temperature": 0.1,
            "do_sample": False,
            "return_full_text": False,
        }
    }

    start = time.time()
    response = requests.post(url, headers=headers, json=payload, timeout=120)
    latency = (time.time() - start) * 1000

    if response.status_code != 200:
        return {"issue_type": "api_error", "order_id": None,
                "response": f"HTTP {response.status_code}: {response.text[:200]}"}, latency

    data = response.json()

    # Extract generated text
    if isinstance(data, list) and len(data) > 0:
        text = data[0].get("generated_text", "")
    else:
        text = str(data)

    text = text.strip()

    try:
        start_idx = text.find("{")
        end_idx   = text.rfind("}") + 1
        if start_idx != -1 and end_idx > start_idx:
            text = text[start_idx:end_idx]
        result = json.loads(text)
    except:
        result = {"issue_type": "raw", "order_id": None, "response": text[:300]}

    return result, latency

def call_gpt4o_mini(message):
    key = os.environ.get("OPENAI_KEY", "")
    client = OpenAI(api_key=key)
    start = time.time()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": message}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    latency = (time.time() - start) * 1000
    return json.loads(response.choices[0].message.content), latency

def compare(message):
    if not message.strip():
        return "", "", "", "", "", ""

    try:
        sft_result, sft_latency = call_supportlm(message)
        sft_issue    = sft_result.get("issue_type", "N/A")
        sft_order_id = str(sft_result.get("order_id", "None detected"))
        sft_response = f"[{sft_latency:.0f}ms] {sft_result.get('response', 'N/A')}"
    except Exception as e:
        sft_issue    = "error"
        sft_order_id = "error"
        sft_response = f"ERROR: {str(e)}"

    try:
        gpt_result, gpt_latency = call_gpt4o_mini(message)
        gpt_issue    = gpt_result.get("issue_type", "N/A")
        gpt_order_id = str(gpt_result.get("order_id", "None detected"))
        gpt_response = f"[{gpt_latency:.0f}ms] {gpt_result.get('response', 'N/A')}"
    except Exception as e:
        gpt_issue    = "error"
        gpt_order_id = "error"
        gpt_response = f"ERROR: {str(e)}"

    return sft_issue, sft_order_id, sft_response, gpt_issue, gpt_order_id, gpt_response

EXAMPLES = [
    "bhai mere order ka kya hua yaar 3 din ho gaye ORDER123",
    "payment cut ho gayi order nahi aaya 😤",
    "enna da ithu order vandhe illa bro paisa pochi",
    "return karna hai item damaged tha MYN456",
]

with gr.Blocks(title="SupportLM Demo") as demo:
    gr.Markdown("""
# 🛍️ SupportLM — Indian E-Commerce Support
**Fine-tuned 3B model vs GPT-4o-mini on Hinglish/Tanglish customer support**

| Metric | GPT-4o-mini | SupportLM |
|--------|-------------|-----------|
| Classification accuracy | 74.8% | **79.9%** |
| Order ID extraction | 98.7% | **99.4%** |
| Language match rate | 96.2% | **99.4%** |
    """)

    message_input = gr.Textbox(
        label="Customer message (Hinglish or Tanglish)",
        placeholder="bhai mere order ka kya hua yaar...",
        lines=3
    )

    submit_btn = gr.Button("🚀 Compare Models", variant="primary")
    gr.Examples(examples=EXAMPLES, inputs=message_input)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🤖 SupportLM (Fine-tuned 3B)")
            sft_issue    = gr.Textbox(label="Issue Type")
            sft_order_id = gr.Textbox(label="Order ID")
            sft_response = gr.Textbox(label="Response [latency]", lines=5)

        with gr.Column():
            gr.Markdown("### 💬 GPT-4o-mini")
            gpt_issue    = gr.Textbox(label="Issue Type")
            gpt_order_id = gr.Textbox(label="Order ID")
            gpt_response = gr.Textbox(label="Response [latency]", lines=5)

    submit_btn.click(
        fn=compare,
        inputs=message_input,
        outputs=[sft_issue, sft_order_id, sft_response,
                 gpt_issue, gpt_order_id, gpt_response]
    )

    gr.Markdown(
        "Model: [GuruVarshini/supportlm-qwen2.5-3b](https://huggingface.co/GuruVarshini/supportlm-qwen2.5-3b) "
        "| Trained with QLoRA + DPO on Google Colab T4 (free tier)"
    )

demo.launch()
'''

REQUIREMENTS = """openai
gradio
requests
"""

with open('/content/drive/MyDrive/SupportLM/app.py', 'w') as f:
    f.write(APP_CODE)

with open('/content/drive/MyDrive/SupportLM/requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)

print("Files saved. Upload both to your Space.")
print("Also click 'Ask for provider support' on your model page.")

Files saved. Upload both to your Space.
Also click 'Ask for provider support' on your model page.


In [ ]:
HTML_APP = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>SupportLM Demo</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body { font-family: system-ui, sans-serif; background: #0f0f0f; color: #e0e0e0; padding: 20px; }
        h1 { color: #ff6b35; margin-bottom: 8px; }
        .subtitle { color: #aaa; margin-bottom: 20px; font-size: 14px; }
        table { border-collapse: collapse; margin-bottom: 20px; width: 100%; max-width: 600px; }
        th, td { border: 1px solid #333; padding: 8px 12px; text-align: left; font-size: 13px; }
        th { background: #1a1a1a; }
        .bold { font-weight: bold; color: #4caf50; }
        .config { background: #1a1a1a; padding: 15px; border-radius: 8px; margin-bottom: 20px; }
        .config label { display: block; margin-bottom: 6px; font-size: 13px; color: #aaa; }
        .config input { width: 100%; padding: 8px; background: #2a2a2a; border: 1px solid #444; border-radius: 4px; color: #e0e0e0; font-size: 13px; }
        .input-area { margin-bottom: 16px; }
        textarea { width: 100%; padding: 12px; background: #1a1a1a; border: 1px solid #444; border-radius: 8px; color: #e0e0e0; font-size: 14px; resize: vertical; min-height: 80px; }
        button { background: #ff6b35; color: white; border: none; padding: 12px 24px; border-radius: 8px; font-size: 15px; cursor: pointer; width: 100%; margin-bottom: 20px; }
        button:hover { background: #e55a25; }
        button:disabled { background: #555; cursor: not-allowed; }
        .examples { margin-bottom: 20px; }
        .examples span { font-size: 12px; color: #aaa; margin-right: 8px; }
        .example-btn { background: #2a2a2a; color: #ccc; border: 1px solid #444; padding: 6px 10px; border-radius: 4px; cursor: pointer; font-size: 12px; margin: 3px; width: auto; display: inline-block; }
        .example-btn:hover { background: #3a3a3a; }
        .results { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
        .model-box { background: #1a1a1a; border-radius: 8px; padding: 16px; }
        .model-box h3 { margin-bottom: 12px; color: #ff6b35; font-size: 15px; }
        .field { margin-bottom: 10px; }
        .field label { font-size: 11px; color: #888; display: block; margin-bottom: 3px; }
        .field .value { background: #2a2a2a; padding: 8px; border-radius: 4px; font-size: 13px; min-height: 36px; word-break: break-word; }
        .response-value { min-height: 80px !important; }
        .loading { color: #ff6b35; font-style: italic; }
        .error { color: #f44336; }
        @media (max-width: 600px) { .results { grid-template-columns: 1fr; } }
    </style>
</head>
<body>
    <h1>🛍️ SupportLM Demo</h1>
    <p class="subtitle">Fine-tuned 3B model vs GPT-4o-mini on Hinglish/Tanglish customer support</p>

    <table>
        <tr><th>Metric</th><th>GPT-4o-mini</th><th>SupportLM</th></tr>
        <tr><td>Classification accuracy</td><td>74.8%</td><td class="bold">79.9%</td></tr>
        <tr><td>Order ID extraction</td><td>98.7%</td><td class="bold">99.4%</td></tr>
        <tr><td>Language match rate</td><td>96.2%</td><td class="bold">99.4%</td></tr>
    </table>

    <div class="config">
        <p style="font-size:12px;color:#888;margin-bottom:10px;">Enter your API keys (stored only in your browser, never sent to any server)</p>
        <label>HuggingFace Token (for SupportLM)</label>
        <input type="password" id="hf_token" placeholder="hf_..." />
        <br><br>
        <label>OpenAI API Key (for GPT-4o-mini)</label>
        <input type="password" id="openai_key" placeholder="sk-..." />
    </div>

    <div class="examples">
        <span>Examples:</span>
        <button class="example-btn" onclick="setExample(this)">bhai mere order ka kya hua yaar ORDER123</button>
        <button class="example-btn" onclick="setExample(this)">payment cut ho gayi order nahi aaya 😤</button>
        <button class="example-btn" onclick="setExample(this)">enna da ithu order vandhe illa bro</button>
        <button class="example-btn" onclick="setExample(this)">return karna hai item damaged tha MYN456</button>
    </div>

    <div class="input-area">
        <textarea id="message" placeholder="bhai mere order ka kya hua yaar..."></textarea>
    </div>

    <button onclick="compare()" id="submit-btn">🚀 Compare Models</button>

    <div class="results">
        <div class="model-box">
            <h3>🤖 SupportLM (Fine-tuned 3B)</h3>
            <div class="field"><label>Issue Type</label><div class="value" id="sft-issue">—</div></div>
            <div class="field"><label>Order ID</label><div class="value" id="sft-order">—</div></div>
            <div class="field"><label>Response [latency]</label><div class="value response-value" id="sft-response">—</div></div>
        </div>
        <div class="model-box">
            <h3>💬 GPT-4o-mini</h3>
            <div class="field"><label>Issue Type</label><div class="value" id="gpt-issue">—</div></div>
            <div class="field"><label>Order ID</label><div class="value" id="gpt-order">—</div></div>
            <div class="field"><label>Response [latency]</label><div class="value response-value" id="gpt-response">—</div></div>
        </div>
    </div>

    <p style="margin-top:20px;font-size:12px;color:#555;">
        Model: <a href="https://huggingface.co/GuruVarshini/supportlm-qwen2.5-3b" style="color:#ff6b35;">GuruVarshini/supportlm-qwen2.5-3b</a>
        | Trained with QLoRA + DPO on Google Colab T4 (free tier)
    </p>

    <script>
        const SYSTEM_PROMPT = "You are a customer support agent for an Indian e-commerce platform. Given a customer message, respond with a JSON object containing issue_type, order_id, and response.";
        const MODEL_ID = "GuruVarshini/supportlm-qwen2.5-3b";

        function setExample(btn) {
            document.getElementById('message').value = btn.innerText;
        }

        function parseJSON(text) {
            try {
                const start = text.indexOf('{');
                const end = text.lastIndexOf('}') + 1;
                if (start !== -1 && end > start) {
                    return JSON.parse(text.slice(start, end));
                }
            } catch(e) {}
            return { issue_type: 'parse_error', order_id: null, response: text.slice(0, 200) };
        }

        async function callSupportLM(message, token) {
            const prompt = `<|im_start|>system\\n${SYSTEM_PROMPT}<|im_end|>\\n<|im_start|>user\\n${message}<|im_end|>\\n<|im_start|>assistant\\n`;
            const start = Date.now();
            const resp = await fetch(`https://api-inference.huggingface.co/models/${MODEL_ID}`, {
                method: 'POST',
                headers: { 'Authorization': `Bearer ${token}`, 'Content-Type': 'application/json' },
                body: JSON.stringify({
                    inputs: prompt,
                    parameters: { max_new_tokens: 200, temperature: 0.1, do_sample: false, return_full_text: false }
                })
            });
            const latency = Date.now() - start;
            const data = await resp.json();
            if (!resp.ok) throw new Error(JSON.stringify(data));
            const text = Array.isArray(data) ? data[0].generated_text : JSON.stringify(data);
            return { result: parseJSON(text), latency };
        }

        async function callGPT4oMini(message, key) {
            const start = Date.now();
            const resp = await fetch('https://api.openai.com/v1/chat/completions', {
                method: 'POST',
                headers: { 'Authorization': `Bearer ${key}`, 'Content-Type': 'application/json' },
                body: JSON.stringify({
                    model: 'gpt-4o-mini',
                    messages: [
                        { role: 'system', content: SYSTEM_PROMPT },
                        { role: 'user', content: message }
                    ],
                    temperature: 0,
                    response_format: { type: 'json_object' }
                })
            });
            const latency = Date.now() - start;
            const data = await resp.json();
            if (!resp.ok) throw new Error(JSON.stringify(data));
            return { result: JSON.parse(data.choices[0].message.content), latency };
        }

        async function compare() {
            const message = document.getElementById('message').value.trim();
            const hfToken = document.getElementById('hf_token').value.trim();
            const openaiKey = document.getElementById('openai_key').value.trim();

            if (!message) { alert('Please enter a message'); return; }

            const btn = document.getElementById('submit-btn');
            btn.disabled = true;
            btn.innerText = 'Running...';

            ['sft-issue','sft-order','sft-response','gpt-issue','gpt-order','gpt-response']
                .forEach(id => { document.getElementById(id).innerHTML = '<span class="loading">Loading...</span>'; });

            const [sftRes, gptRes] = await Promise.allSettled([
                callSupportLM(message, hfToken),
                callGPT4oMini(message, openaiKey)
            ]);

            if (sftRes.status === 'fulfilled') {
                const { result, latency } = sftRes.value;
                document.getElementById('sft-issue').innerText = result.issue_type || 'N/A';
                document.getElementById('sft-order').innerText = result.order_id || 'None detected';
                document.getElementById('sft-response').innerText = `[${latency}ms] ${result.response || 'N/A'}`;
            } else {
                document.getElementById('sft-issue').innerHTML = '<span class="error">error</span>';
                document.getElementById('sft-order').innerHTML = '<span class="error">error</span>';
                document.getElementById('sft-response').innerHTML = `<span class="error">${sftRes.reason}</span>`;
            }

            if (gptRes.status === 'fulfilled') {
                const { result, latency } = gptRes.value;
                document.getElementById('gpt-issue').innerText = result.issue_type || 'N/A';
                document.getElementById('gpt-order').innerText = result.order_id || 'None detected';
                document.getElementById('gpt-response').innerText = `[${latency}ms] ${result.response || 'N/A'}`;
            } else {
                document.getElementById('gpt-issue').innerHTML = '<span class="error">error</span>';
                document.getElementById('gpt-order').innerHTML = '<span class="error">error</span>';
                document.getElementById('gpt-response').innerHTML = `<span class="error">${gptRes.reason}</span>`;
            }

            btn.disabled = false;
            btn.innerText = '🚀 Compare Models';
        }
    </script>
</body>
</html>"""

with open('/content/drive/MyDrive/SupportLM/index.html', 'w') as f:
    f.write(HTML_APP)

print("index.html saved to Drive")
print("Upload this single file to your Static HuggingFace Space")

index.html saved to Drive
Upload this single file to your Static HuggingFace Space


In [4]:
import json

# Load the notebook
notebook_path = "/content/drive/MyDrive/SupportLM/SupportLM.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove the problematic widget metadata
# This is what causes "Invalid Notebook" on GitHub
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("Removed widget metadata")
else:
    print("No widget metadata found")

# Also clean widget output from each cell
for cell in nb.get("cells", []):
    if "outputs" in cell:
        cell["outputs"] = [
            out for out in cell["outputs"]
            if out.get("output_type") != "display_data" or
            "application/vnd.jupyter.widget-view+json" not in out.get("data", {})
        ]

# Save cleaned notebook
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print("Notebook cleaned and saved")
print("Download it from Drive and push to GitHub")

Removed widget metadata
Notebook cleaned and saved
Download it from Drive and push to GitHub
